# rStar-Math Integration Examples

This notebook demonstrates how to integrate rStar-Math with various frameworks and platforms.

In [ ]:
import os
from typing import Dict, List
from src.core.mcts import MCTS
from src.core.ppm import ProcessPreferenceModel
from src.models.model_interface import ModelFactory

## 1. Basic Integration

First, let's see how to use rStar-Math directly.

In [ ]:
# Initialize components
mcts = MCTS.from_config_file('config/default.json')
ppm = ProcessPreferenceModel.from_config_file('config/default.json')
model = ModelFactory.create_model(
    'openai',
    os.getenv('OPENAI_API_KEY'),
    'config/default.json'
)

# Solve a problem
problem = "What is the derivative of f(x) = x^2 + 3x?"
action, trajectory = mcts.search(problem)

# Print solution steps with confidence scores
for step in trajectory:
    confidence = ppm.evaluate_step(step['state'], model)
    print(f"Step: {step['state']}")
    print(f"Confidence: {confidence:.2f}\n")

## 2. LangChain Integration

Here's how to use rStar-Math with LangChain.

In [ ]:
from langchain.llms import OpenAI
from examples.langchain_integration import RStarMathChain

# Initialize LangChain components
llm = OpenAI(temperature=0.7)
chain = RStarMathChain(
    llm=llm,
    api_key=os.getenv('OPENAI_API_KEY')
)

# Solve problem
result = chain.run("What is 2 + 2?")

print("Direct Solution:")
print(result['direct_solution'])
print(f"Score: {result['direct_score']:.2f}\n")

print("Enhanced Solution:")
print(result['enhanced_solution'])
print(f"Score: {result['enhanced_score']:.2f}\n")

print(f"Improvement: {result['improvement']:.2%}")

## 3. Rasa Integration

Example of using rStar-Math in a Rasa chatbot.

In [ ]:
from rasa_sdk import Action
from examples.rasa_integration import RStarMathAction

# Create mock Rasa components for demonstration
class MockDispatcher:
    def utter_message(self, text: str):
        print(f"Bot: {text}")

class MockTracker:
    def __init__(self, text: str):
        self.latest_message = {"text": text}

# Initialize action
action = RStarMathAction()

# Simulate conversation
async def simulate_conversation():
    dispatcher = MockDispatcher()
    tracker = MockTracker("What is the derivative of x^2?")
    
    print(f"User: {tracker.latest_message['text']}")
    await action.run(dispatcher, tracker, {})

# Run simulation
import asyncio
asyncio.run(simulate_conversation())

## 4. Custom Integration

Example of creating a custom integration with rStar-Math.

In [ ]:
class MathTutor:
    def __init__(self, api_key: str):
        self.mcts = MCTS.from_config_file('config/default.json')
        self.ppm = ProcessPreferenceModel.from_config_file('config/default.json')
        self.model = ModelFactory.create_model('openai', api_key, 'config/default.json')
        
    def solve_with_explanation(self, problem: str) -> Dict:
        # Get solution using rStar-Math
        action, trajectory = self.mcts.search(problem)
        
        # Format steps with confidence scores
        steps = []
        total_confidence = 0.0
        
        for step in trajectory:
            confidence = self.ppm.evaluate_step(step['state'], self.model)
            steps.append({
                'explanation': step['state'],
                'confidence': confidence
            })
            total_confidence += confidence
            
        return {
            'steps': steps,
            'average_confidence': total_confidence / len(steps) if steps else 0.0
        }

# Test the custom integration
tutor = MathTutor(os.getenv('OPENAI_API_KEY'))
result = tutor.solve_with_explanation("Solve for x: 2x + 3 = 7")

print("Solution Steps:")
for i, step in enumerate(result['steps'], 1):
    print(f"\nStep {i}:")
    print(f"Explanation: {step['explanation']}")
    print(f"Confidence: {step['confidence']:.2f}")

print(f"\nOverall Confidence: {result['average_confidence']:.2f}")